In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
    )
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV


In [6]:
df=pd.read_csv('data/ready_to_train.csv')

In [24]:
X = df.drop(columns=['TARGET'])
y = df['TARGET']

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_features = X.select_dtypes(exclude='object').columns
cat_features = X.select_dtypes(include='object').columns

num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ]
)

In [28]:
X_train,X_test,y_train,y_test=train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [29]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [31]:
def evaluate_model(model, X_train, X_test):
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]

    print("Accuracy:",
          accuracy_score(y_test, pred))

    print("ROC AUC:",
          roc_auc_score(y_test, proba))

    print(classification_report(
        y_test,
        pred
    ))

In [30]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)

lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [32]:
evaluate_model(lr,X_train, X_test)

Accuracy: 0.6903240492333708
ROC AUC: 0.7487261479477286
              precision    recall  f1-score   support

           0       0.96      0.69      0.80     56538
           1       0.16      0.68      0.26      4965

    accuracy                           0.69     61503
   macro avg       0.56      0.68      0.53     61503
weighted avg       0.90      0.69      0.76     61503



In [18]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)

evaluate_model(
    rf,
    X_train,
    X_test
)

Accuracy: 0.919337268100743
ROC AUC: 0.7386803685083141
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.64      0.00      0.00      4965

    accuracy                           0.92     61503
   macro avg       0.78      0.50      0.48     61503
weighted avg       0.90      0.92      0.88     61503



In [33]:
from xgboost import XGBClassifier

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)
evaluate_model(xgb,X_train, X_test)

Accuracy: 0.7179324585792562
ROC AUC: 0.7685961089471431
              precision    recall  f1-score   support

           0       0.96      0.72      0.82     56538
           1       0.18      0.68      0.28      4965

    accuracy                           0.72     61503
   macro avg       0.57      0.70      0.55     61503
weighted avg       0.90      0.72      0.78     61503



In [34]:
proba = xgb.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

In [35]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))
print("ROC AUC :", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy : 0.7179324585792562
Precision: 0.17615984099586798
Recall : 0.6783484390735146
F1 Score : 0.2796877595083873
ROC AUC : 0.7685961089471431
              precision    recall  f1-score   support

           0       0.96      0.72      0.82     56538
           1       0.18      0.68      0.28      4965

    accuracy                           0.72     61503
   macro avg       0.57      0.70      0.55     61503
weighted avg       0.90      0.72      0.78     61503



In [36]:
import numpy as np
from sklearn.metrics import f1_score

best_threshold = 0
best_f1 = 0

for t in np.arange(0.1, 0.91, 0.01):
    pred = (proba >= t).astype(int)
    f1 = f1_score(y_test, pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

Best Threshold: 0.6599999999999997
Best F1: 0.3167283997278706


In [37]:
best_threshold=0.66
proba = xgb.predict_proba(X_test)[:, 1]
pred = (proba >= best_threshold).astype(int)

In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))
print("ROC AUC :", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy : 0.8530315594361251
Precision: 0.25350919651500486
Recall : 0.4219536757301108
F1 Score : 0.3167283997278706
ROC AUC : 0.7685961089471431
              precision    recall  f1-score   support

           0       0.95      0.89      0.92     56538
           1       0.25      0.42      0.32      4965

    accuracy                           0.85     61503
   macro avg       0.60      0.66      0.62     61503
weighted avg       0.89      0.85      0.87     61503



In [40]:
pred_df=pd.DataFrame({'Actual Value':y_test,'Predicted Value':pred,'Difference':y_test-pred})
pred_df

,Actual Value,Predicted Value,Difference
256571,0,0,0
191493,0,0,0
103497,0,1,-1
130646,0,0,0
211898,0,0,0
...,...,...,...
16213,0,0,0
294620,0,0,0
234384,0,0,0
149027,1,0,1


In [41]:
import pickle

with open("xgb_model.pkl", "wb") as f:
    pickle.dump(xgb, f)

In [42]:
with open("threshold.pkl", "wb") as f:
    pickle.dump(best_threshold, f)

In [43]:
with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)